# EcoGuard — Kaggle Notebook

**Multi-agent environmental companion** (demo + real API integration examples)

**What this notebook contains:**

- Self-contained agent implementations (Environment, Safety, Community, Planner, Reporter)
- Real API integration examples for **OpenWeather (One Call / Air Pollution)** and **WAQI (aqicn)**
- Fallback mock data for reproducible runs on Kaggle (no API keys required)

_Created: 2025-11-29 17:32 UTC_

## Quick instructions

1. Run the cells in order. The notebook uses mock data by default so it works on Kaggle without credentials.
2. To enable real APIs, set your API keys (OpenWeather `OWM_API_KEY` and WAQI `WAQI_TOKEN`) using one of the methods below.
3. If you plan to publish on Kaggle, use Kaggle Secrets (see a cell below) or create a small dataset with a `.env` file.


In [ ]:
# Install any missing packages (uncomment if running locally)
# On Kaggle the common libs are usually preinstalled; run these if needed.
# !pip install python-dotenv requests pandas


In [ ]:
# Imports and API key loading helpers
import os
import json
import requests
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List
from dataclasses import dataclass, field

# Load .env if present (local dev)
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Try environment variables first (recommended)
OWM_API_KEY = os.getenv('OWM_API_KEY')        # OpenWeather API key (One Call / Air Pollution)
WAQI_TOKEN = os.getenv('WAQI_TOKEN')         # WAQI / AQICN token

print('OpenWeather key loaded:', bool(OWM_API_KEY))
print('WAQI token loaded:', bool(WAQI_TOKEN))

## Where to store API keys

- **Locally:** create a `.env` file with `OWM_API_KEY=your_key` and `WAQI_TOKEN=your_token`, then use `python-dotenv`.
- **Kaggle Notebook:**
  - Use **Secrets** (Kaggle UI → Settings → Secrets) to add environment variables `OWM_API_KEY` and `WAQI_TOKEN`.
  - Or upload a small private dataset that contains a `.env` file and read it at runtime (less recommended).

**Security note:** never commit API keys to GitHub. Use environment variables or Kaggle Secrets.


In [ ]:
# OpenWeather and WAQI integration helpers
import math

def call_openweather_current(lat: float, lon: float, api_key: str) -> Dict[str, Any]:
    """Call OpenWeather Current Weather API (or One Call / Air Pollution) to get weather and optionally air pollution."""
    base = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'lat': lat, 'lon': lon, 'appid': api_key, 'units': 'metric'}
    # Accept both city->coord or direct coords: this helper expects coords
    resp = requests.get(base, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

def call_openweather_air_pollution(lat: float, lon: float, api_key: str) -> Dict[str, Any]:
    base = 'https://api.openweathermap.org/data/2.5/air_pollution'
    params = {'lat': lat, 'lon': lon, 'appid': api_key}
    resp = requests.get(base, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

def call_waqi_city(city: str, token: str) -> Dict[str, Any]:
    # WAQI: city feed endpoint: https://api.waqi.info/feed/{city}/?token={token}
    base = f'https://api.waqi.info/feed/{city}/'
    params = {'token': token}
    resp = requests.get(base, params=params, timeout=12)
    resp.raise_for_status()
    return resp.json()

def geocode_city_to_coords(city: str, api_key: str) -> Optional[Dict[str, float]]:
    # OpenWeather provides geocoding API
    base = 'http://api.openweathermap.org/geo/1.0/direct'
    params = {'q': city, 'limit': 1, 'appid': api_key}
    resp = requests.get(base, params=params, timeout=8)
    resp.raise_for_status()
    items = resp.json()
    if not items:
        return None
    return {'lat': items[0]['lat'], 'lon': items[0]['lon']}


In [ ]:
# Agent implementations (EnvironmentAgent uses real APIs if keys present, otherwise mock)
import random
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class EnvironmentData:
    timestamp: datetime
    city: str
    aqi: int = 0
    pm25: float = 0.0
    pm10: float = 0.0
    uv_index: float = 0.0
    temperature_c: float = 0.0
    humidity_pct: int = 0
    rainfall_mm: float = 0.0
    pollen_level: str = 'moderate'
    weather_alerts: list = field(default_factory=list)

    def to_dict(self):
        return {**self.__dict__}

class EnvironmentAgent:
    def __init__(self, owm_key: Optional[str]=None, waqi_token: Optional[str]=None):
        self.owm_key = owm_key
        self.waqi_token = waqi_token

    def fetch_current(self, city: str, date: Optional[datetime]=None) -> EnvironmentData:
        if date is None:
            date = datetime.utcnow()
        # try real APIs if keys present
        if self.owm_key:
            coords = geocode_city_to_coords(city, self.owm_key)
            if coords:
                try:
                    w = call_openweather_current(coords['lat'], coords['lon'], self.owm_key)
                    # basic extraction
                    temp = w.get('main', {}).get('temp')
                    humidity = w.get('main', {}).get('humidity')
                    # UV & detailed air pollution require other endpoints; we will fetch air pollution below
                    aqi = None
                    if self.owm_key:
                        try:
                            ap = call_openweather_air_pollution(coords['lat'], coords['lon'], self.owm_key)
                            # OpenWeather returns list in 'list'[0]['main']['aqi'] (scale 1-5)
                            if 'list' in ap and ap['list']:
                                owm_aqi_index = ap['list'][0]['main'].get('aqi')
                                # convert OWM aqi index (1-5) to rough US AQI scale for readability
                                aqi = int(owm_aqi_index * 50) if owm_aqi_index else None
                                # also try pm2_5
                                comps = ap['list'][0].get('components', {})
                                pm25 = comps.get('pm2_5')
                                pm10 = comps.get('pm10')
                            else:
                                pm25 = pm10 = None
                        except Exception as e:
                            aqi = None
                            pm25 = pm10 = None
                    else:
                        pm25 = pm10 = None

                    # try WAQI if token provided for a more standard AQI
                    if self.waqi_token:
                        try:
                            waqi = call_waqi_city(city, self.waqi_token)
                            if waqi.get('status') == 'ok' and 'data' in waqi:
                                # WAQI gives 'aqi' numeric
                                aqi = int(waqi['data'].get('aqi') or (aqi or 0))
                                iaqi = waqi['data'].get('iaqi', {})
                                pm25 = iaqi.get('pm25', {}).get('v', pm25)
                                pm10 = iaqi.get('pm10', {}).get('v', pm10)
                        except Exception as e:
                            pass

                    # UV: OpenWeather has separate UV/OneCall endpoints (omitted to reduce calls); mock UV
                    uv = round(max(0, min(11, random.gauss(6,1.5))),1)

                    env = EnvironmentData(timestamp=date, city=city,
                                          aqi=int(aqi or (random.randint(40,150))),
                                          pm25=round(pm25 or random.uniform(5,40),1),
                                          pm10=round(pm10 or random.uniform(10,60),1),
                                          uv_index=uv,
                                          temperature_c=float(temp if temp is not None else round(random.gauss(28,6),1)),
                                          humidity_pct=int(humidity if humidity is not None else random.randint(30,80)),
                                          rainfall_mm=round(random.gauss(1,3),1),
                                          pollen_level=random.choice(['low','moderate','high']))
                    return env
                except Exception as e:
                    # fallback to mock
                    pass

        # Mock deterministic fallback (seed by city+date)
        seed = sum(ord(c) for c in (city or 'default')) + date.day + date.month
        rnd = random.Random(seed)
        aqi = int(max(10, min(500, rnd.gauss(120, 60))))
        return EnvironmentData(timestamp=date, city=city, aqi=aqi,
                               pm25=round(max(1, rnd.gauss(aqi/4,5)),1),
                               pm10=round(max(5, rnd.gauss(aqi/3,6)),1),
                               uv_index=round(max(0, rnd.gauss(6,2)),1),
                               temperature_c=round(rnd.gauss(28,6),1),
                               humidity_pct=int(max(10, min(100, rnd.gauss(65,20)))),
                               rainfall_mm=round(max(0, rnd.gauss(1,5)),1),
                               pollen_level=rnd.choice(['low','moderate','high']))

# Other agents (re-use from previous design)
class SafetyAgent:
    def advise(self, env: EnvironmentData, user: Dict[str,Any]) -> Dict[str,Any]:
        tips=[]; reasons=[]
        aqi = env.aqi
        if aqi<=50: tips.append('Air quality is good — safe for outdoor activities.')
        elif aqi<=100: tips.append('Air quality is moderate — sensitive individuals should take caution.')
        elif aqi<=200: tips.append('Air quality is unhealthy for sensitive groups — limit prolonged exertion.'); reasons.append('aqi_unhealthy_sensitive')
        elif aqi<=300: tips.append('Air quality is unhealthy — avoid outdoor exertion and wear a mask if needed.'); reasons.append('aqi_unhealthy')
        else: tips.append('Air quality is hazardous — stay indoors and use air purifiers if available.'); reasons.append('aqi_hazardous')
        if user.get('has_asthma'): tips.append('You reported asthma — keep inhaler nearby and avoid peak pollution hours.')
        if env.uv_index>=8: tips.append('Very high UV — use sunscreen and protective clothing.')
        if env.temperature_c>=38: tips.append('High temperature — hydrate and avoid strenuous activity.')
        if env.pollen_level=='high': tips.append('High pollen — allergy sufferers should limit outdoor exposure.')
        return {'tips': tips, 'reasons': reasons, 'short_summary': tips[0] if tips else 'Conditions look normal.'}

class CommunityAgent:
    ACTIONS = ['Use a reusable bottle','Avoid single-use plastics','Plant a small herb','Share a local eco-tip']
    def suggest_actions(self, env: EnvironmentData, n=2):
        rnd = random.Random(env.aqi + int(env.temperature_c))
        out = []
        if env.aqi>150: out.append('Avoid burning waste at home — it raises local pollution.')
        for _ in range(max(0,n-len(out))):
            out.append(rnd.choice(self.ACTIONS))
        return out

class PlannerAgent:
    def plan_week(self, city: str):
        from datetime import datetime, timedelta
        start = datetime.utcnow()
        plan=[] 
        for d in range(7):
            date = start + timedelta(days=d)
            env = EnvironmentAgent().fetch_current(city, date=date)
            mood = 'Good for outdoor activity' if env.aqi<=100 and env.uv_index<=7 and env.temperature_c<36 else 'Prefer indoor or light outdoor activity'
            plant = 'Water plants' if env.rainfall_mm<3 else 'Skip watering (rain expected)'
            plan.append({'date': date.date().isoformat(),'mood':mood,'plant_care':plant,'aqi':env.aqi})
        return plan

class ReporterAgent:
    def generate_report(self, env: EnvironmentData, safety, actions, user):
        greeting = f"Good morning, {user.get('name','friend')}!"
        lines=[greeting,f"Today in {env.city}: {safety['short_summary']}", '', 'Quick facts:',
               f" • AQI: {env.aqi}", f" • Temp: {env.temperature_c} °C", f" • UV: {env.uv_index}"]
        if env.weather_alerts: lines.append(' • Alerts: '+', '.join(env.weather_alerts))
        lines.append(''); lines.append('Personalized tips:')
        for t in safety['tips'][:6]: lines.append(' • '+t)
        lines.append(''); lines.append('Community actions you can try today:')
        for a in actions: lines.append(' • '+a)
        return {'timestamp':datetime.utcnow().isoformat(),'city':env.city,'summary_text':'\n'.join(lines),'env':env.to_dict(),'safety':safety,'actions':actions,'user':user}


In [ ]:
# Demo: run the pipeline for an example profile (uses real APIs if keys are set)
profile = {'name':'Demo User','age':32,'city':'Mumbai','has_asthma':False,'prefers_outdoor_exercise':True}

env_agent = EnvironmentAgent(owm_key=OWM_API_KEY, waqi_token=WAQI_TOKEN)
safety_agent = SafetyAgent()
community_agent = CommunityAgent()
planner_agent = PlannerAgent()
reporter_agent = ReporterAgent()

env = env_agent.fetch_current(profile['city'])
safety = safety_agent.advise(env, profile)
actions = community_agent.suggest_actions(env)
week = planner_agent.plan_week(profile['city'])
report = reporter_agent.generate_report(env, safety, actions, profile)

print(report['summary_text'])

# show week plan table if pandas is available
try:
    import pandas as pd
    df = pd.DataFrame(week)
    display(df)
except Exception:
    pass


In [ ]:
# Simple evaluation: simulate profiles and count strong 'stay indoors' recommendations
def evaluate_sim(n=100):
    indoors = 0
    high_aqi = 0
    for i in range(n):
        user = {'name':f'User{i}','age':20+i%50,'city':'Delhi' if i%3==0 else 'Mumbai','has_asthma':(i%7==0),'prefers_outdoor_exercise':bool(i%2)}
        env = EnvironmentAgent(owm_key=OWM_API_KEY, waqi_token=WAQI_TOKEN).fetch_current(user['city'], date=datetime.utcnow())
        s = SafetyAgent().advise(env, user)
        if any('stay indoors' in tip.lower() or 'stay indoors' in s.get('short_summary','').lower() for tip in s['tips']):
            indoors += 1
        if env.aqi >= 151:
            high_aqi += 1
    return {'simulated':n,'indoors':indoors,'high_aqi':high_aqi}

res = evaluate_sim(50)
print(res)


## Final notes

- This notebook includes **real API examples** and **mock fallbacks** so it runs on Kaggle without keys.
- If you enable `OWM_API_KEY` and/or `WAQI_TOKEN`, the notebook will call the live APIs and enrich the EnvironmentAgent output.
- To publish: include this notebook in your Kaggle project and use Kaggle Secrets for the keys.

---

**Sources:** OpenWeather API docs (Current Weather, One Call, Air Pollution) and WAQI API docs were used to build the integration examples. See the notebook header in the submission for links.
